# Convolutional Neural Network NILM

## Imports

In [23]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Hide Warnings

## Directories

In [2]:
base_dir = os.getcwd()
processed_data_dir = os.path.join(base_dir, 'processed_data')

results_dir = os.path.join(base_dir, 'results')
os.makedirs(results_dir, exist_ok=True)

## Load Data

In [9]:
processed_data_filepath = os.path.join(processed_data_dir, 'ampds_dataset.npz')
data = np.load(processed_data_filepath)

# Check data
print(f'Dataset Items: {data.files}')
print(f'x_norm shape: {data['x_norm'].shape}')
print(f'y_norm shape: {data['y_norm'].shape}')

# Check index shapes
dataset_names = sorted({
    key.rsplit('_', 1)[0]
    for key in data.files
    if key.endswith(('_inp', '_out'))})

index_shapes = pd.DataFrame([{
    'Dataset': name,
    'Number Input (Frame Start) Indices': data[f'{name}_inp'].shape[0],
    'Number Output (Target) Indices': data[f'{name}_out'].shape[0]
} for name in dataset_names])

display(index_shapes.style.set_caption('Dataset Splits').hide(axis='index'))

Dataset Items: ['appliance_names', 'x_norm', 'y_norm', 'x_min', 'x_max', 'y_min', 'y_max', 'train_inp', 'train_out', 'val_inp', 'val_out', 'test_inp', 'test_out', 'T_limit', 'train_val_test_split', 'window_length', 'stride']
x_norm shape: (1000, 6)
y_norm shape: (1000, 22, 4)


Dataset,Number Input (Frame Start) Indices,Number Output (Target) Indices
test,83,83
train,492,492
val,164,164


In [21]:
p_agg_norm = data['x_norm'][:, 2]  # P-agg
q_agg_norm = data['x_norm'][:, 3]  # Q-agg
p_apps = data['y_norm'][:, 1]

print(f'P-agg shape: {p_agg_norm.shape}')
print(f'Q-agg shape: {q_agg_norm.shape}')
print(f'P-apps shape: {p_apps.shape}')

P-agg shape: (1000,)
Q-agg shape: (1000,)
P-apps shape: (1000, 4)


In [22]:
cnn_data = {
    'p_agg_norm': p_agg_norm,
    'q_agg_norm': q_agg_norm,
    'p_apps': p_apps,
    'window_length': data['window_length'],
    'train_frame_start_indices': data['train_inp'],
    'val_frame_start_indices': data['val_inp'],
    'test_frame_start_indices': data['test_inp']}

## Sample Generation

### PQ Signature

We replicate the CNN method describes in [citation]. This uses a sequence-to-point model, taking a window of time-series data and input and predicting the target value at one timestamp, typically the window's center. To represent the data in a form suitable for a 2D CNN, the method converts the real-power series (P) and reactive-power series (Q) into a two-dimensional P-Q signature matrix. Each element is defined as $S_{xy} = \sqrt{P_x^2 + Q_y^2}$. The first columns represents the real power alone, and the first row represents the reactive power alone.

In [10]:
def make_pq_image(p, q):
    
    S = np.zeros((p.size + 1, q.size + 1))
    S[1:, 0] = np.abs(p)
    S[0, 1:] = np.abs(q)
    S[1:, 1:] = np.hypot(p[:, None], q[None, :])
    return S

We apply a 2D fast Fourier transform (FFT) to the P-Q matrix and use its magnitude. The FFT represents the image in terms of spatial frequencies, showing how quickly patterns change across rows and columns. This may reveal appliance characteristics or operating states that are not easy to see in the original image.

In [11]:
def fft(S): return np.abs(np.fft.fft2(S))

To create each sample, we stack the original time-domain P-Q image and its FFT magnitude representation.

In [16]:
def create_sample(frame_start_idx, p_inp_norm, q_inp_norm, p_out_norm, window_length):
    p = p_inp_norm[frame_start_idx : frame_start_idx + window_length]
    q = q_inp_norm[frame_start_idx : frame_start_idx + window_length]
    center_timestep = frame_start_idx + window_length // 2
    target = p_out_norm[center_timestep]
    S = make_pq_image(p, q)
    S_fft = fft(S)
    return (S, S_fft), target

In [17]:
def create_batch(frame_start_idx_list, p_inp_norm, q_inp_norm, p_out_norm, window_length):
    sample_list = []
    for i in frame_start_idx_list:
        sample = create_sample(i, p_inp_norm, q_inp_norm, p_out_norm, window_length)
        sample_list.append(sample)
    return sample_list

## Model Architecture

The CNN model uses a two-branch architecture. the time-domain branch takes the P-Q signature as input, while the frequency-domain branch takes the FFT as input. Each branch has its own convolutional filters and produces a feature vector. These vectors are concatenates before passing through the dense and output layers. 

In [ ]:
def build_branch(input_layer):
        x = layers.Conv2D(30, (10, 10), activation='relu')(input_layer)
        x = layers.Conv2D(30, (8, 8), activation='relu')(x)
        x = layers.Conv2D(40, (6, 6), activation='relu')(x)
        x = layers.Conv2D(50, (5, 5), activation='relu')(x)
        x = layers.Conv2D(50, (5, 5), activation='relu')(x)
        x = layers.Flatten()(x)
        return x

In [ ]:
def create_model_architecture():

    # Inputs
    inp_time = layers.Input(shape=(31, 31, 1))
    inp_freq = layers.Input(shape=(31, 31, 1))

    # Branches
    branch_time = build_branch(inp_time)
    branch_freq = build_branch(inp_freq)
    x = layers.Concatenate()([branch_time, branch_freq])

    # Dense + Output Layers
    x = layers.Dense(1024, activation='relu')(x)
    out = layers.Dense(1)(x)

    # Model
    model = models.Model(inputs=[inp_time, inp_freq], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse')

    return model

## Train Model

The `train_model` function creates a model, trains it in mini-batches, checks its performance on validation data after each epoch, and uses early stopping to limit overfitting.
- Training: Each epoch, we shuffle the training frame indices, create batches, and call `train_on_batch` to update the model weights. We average the batch losses to get the epoch's training loss.
- Validation: We create validation batches and call `test_on_batch`, which measures loss without updating the weights.
- Early stopping and saving: If validation loss improves, it resets the patience counter and saves the model. Otherwise, it increments the counter and stops after five consecutive epochs without improvement.

In [ ]:
def train_model(data, num_epochs, batch_size, model_save_filepath):
    model = create_model_architecture()

    best_val_loss = np.inf
    patience = 5
    patience_counter = 0

    for epoch in tqdm(range(num_epochs), desc='Epochs'):

        # Training
        train_loss = 0.0
        num_train_batches = 0

        # Shuffle each epoch
        perm = np.random.permutation(len(data['train_frame_start_indices']))
        train_frame_start_indices = data['train_frame_start_indices'][perm]
        train_target_indices = data['train_target_indices'][perm]

        for i in tqdm(range(0, len(train_frame_start_indices), batch_size), desc="Training", leave=False):
            batch_frame_start_indices = train_frame_start_indices[i: i + batch_size]
            batch_target_indices = train_target_indices[i: i + batch_size]
            (S, S_fft), target = create_batch(batch_frame_start_indices, data['p_inp_norm'], data['q_inp_norm'], data['p_out_norm'], data['window_length'])

            loss = model.train_on_batch([S, S_fft], target)
            train_loss += loss
            num_train_batches += 1

        train_loss /= num_train_batches

        # Validation
        val_loss = 0.0
        num_val_batches = 0

        for i in tqdm(range(0, len(data['val_frame_start_indices']), batch_size), desc="Validation", leave=False):
            batch_frame_start_indices = data['val_frame_start_indices'][i: i + batch_size]
            batch_target_indices = data['val_target_indices'][i: i + batch_size]
            (S, S_fft), target = create_batch(batch_frame_start_indices, data['p_inp_norm'], data['q_inp_norm'], data['p_out_norm'], data['window_length'])

            loss = model.test_on_batch([S, S_fft], target)
            val_loss += loss
            num_val_batches += 1

        val_loss /= num_val_batches
        print(f"Epoch {epoch + 1}/{num_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            model.save(model_save_filepath)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print('Early Stopping')
                break

    model.save(model_save_filepath)
